# Muse in-app: e-receipts vs Google Play card transactions, and how each store sends receipts

**Questions (team review, 2026-10-07, items 3 and 4)**
1. Do the NIQ Google Play Muse receipts line up with Google Play card transactions for Meta in the Yodlee panels?
2. Do Google Play and Apple both send a receipt for every recurring (renewal) payment? This decides whether e-receipt charges after month 1 are renewals we can see, or only first payments.

**Method**
- Cards: Yodlee F3/F4/F6 card + bank rows since 2026-08-01 whose descriptor is Google Play billing for Meta. Google Play bills every Meta app under `GOOGLE *FACEBOOK` (and variants); there is no Muse-specific descriptor, so Muse can only be isolated by price: in-app Power $19.99-22.00 and Max $99.99-110.00. One range query per source, pruned on `panel_file_created_date`.
- E-receipts: the cached app load (`app/.cache/niq_receipts`), Google Play `charge` receipts.
- Store behaviour: Muse receipts per mailbox, plus a mature benchmark (ChatGPT Plus, Jul-Sep 2026) because Muse's first renewals only fall due from about 2026-10-08.

In [1]:
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd
import snowflake.connector

APP = Path.cwd().parent / "app"
sys.path.insert(0, str(APP))
import sections as S

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)
conn = snowflake.connector.connect(connection_name="default")


def q(sql: str) -> pd.DataFrame:
    df = conn.cursor().execute(sql).fetch_pandas_all()
    df.columns = [c.lower() for c in df.columns]
    return df


START, CARD_THROUGH, LAUNCH = "2026-08-01", "2026-10-03", pd.Timestamp(S.LAUNCH)
BANDS = {"Power": (19.99, 22.00), "Max": (99.99, 110.00)}

## 1. Google Play card transactions for Meta

In [2]:
def gp_sql(src: dict) -> str:
    d = src["desc"]
    return f"""
        SELECT '{src['name']}' AS source, CAST({src['member']} AS VARCHAR) AS member,
               {src['date']}::DATE AS day, ROUND({src['amt']}, 2) AS amount, {d} AS description
        FROM {src['table']}
        WHERE {S._window(src, START, CARD_THROUGH)}
          AND {d} ILIKE '%google%'
          AND ({d} ILIKE '%facebook%' OR {d} ILIKE '%google%meta %' OR {d} ILIKE '%google%muse %')
          -- other 'muse' / 'meta' apps seen on 2026-10-08
          AND NOT ({d} ILIKE '%my muse%' OR {d} ILIKE '%musescore%' OR {d} ILIKE '%museum%' OR {d} ILIKE '%muse brain%')
    """

with ThreadPoolExecutor(6) as pool:
    raw = pd.concat(pool.map(lambda s: q(gp_sql(s)), S.CARD_SOURCES), ignore_index=True)
raw["day"] = pd.to_datetime(raw.day)
raw["amount"] = raw.amount.astype(float)
# Pending and posted copies carry different ids: dedupe as the app does.
gp = raw.drop_duplicates(["member", "day", "amount"]).copy()
gp["band"] = None
for plan, (lo, hi) in BANDS.items():
    gp.loc[gp.amount.between(lo, hi), "band"] = plan
gp["period"] = (gp.day >= LAUNCH).map({False: "pre-launch", True: "post-launch"})
print(len(raw), "rows,", len(gp), "after dedupe")
gp.groupby(["source"]).size()

44707 rows, 43400 after dedupe


source
Yodlee F3 bank    32852
Yodlee F3 card     1747
Yodlee F4 bank     1565
Yodlee F4 card      546
Yodlee F6 bank     6070
Yodlee F6 card      620
dtype: int64

In [3]:
# Amount mix before and after launch: does a $19.99-22 cluster appear at launch?
days = gp.groupby("period").day.nunique()
mix = (gp.assign(amt=gp.amount.round(0)).groupby(["period", "amt"]).size().unstack(0).fillna(0)
         .div(days, axis=1).round(2))
mix[mix.max(axis=1) >= 0.3].sort_index()

period,post-launch,pre-launch
amt,,
-1.0,0.31,0.26
0.0,11.92,10.08
1.0,301.04,250.53
2.0,60.27,50.47
3.0,70.65,60.53
4.0,18.46,12.50
5.0,116.81,97.92
6.0,10.73,8.97
7.0,1.62,1.82


In [4]:
# Daily and weekly in-band Google Play Meta charges vs NIQ Google Play Muse charges.
niq = pd.read_parquet(sorted((APP / ".cache" / "niq_receipts").glob("*.parquet"))[-1])
niq["order_date"] = pd.to_datetime(niq.order_date)
niq_gp = (niq[(niq.store == "Google Play") & (niq.kind == "charge")]
          .groupby("order_date").mailbox_id.nunique().rename("niq_gp_charges"))

cal = pd.date_range(START, CARD_THROUGH)
card_band = gp[gp.band.notna()].groupby("day").member.nunique().reindex(cal, fill_value=0).rename("card_gp_band")
daily = pd.concat([card_band, niq_gp.reindex(cal, fill_value=0)], axis=1)
weekly = daily.groupby(daily.index.to_period("W-SUN").start_time).sum()  # week starting Monday
weekly

,card_gp_band,niq_gp_charges
2026-07-27,3,0
2026-08-03,35,0
2026-08-10,41,0
2026-08-17,36,0
2026-08-24,42,0
2026-08-31,37,0
2026-09-07,42,1
2026-09-14,45,7
2026-09-21,50,27
2026-09-28,62,29


In [5]:
pre = daily[daily.index < LAUNCH].card_gp_band.mean()
post = daily[daily.index >= LAUNCH]
w_post = weekly[weekly.index >= LAUNCH]
print(f"card in-band/day: pre-launch {pre:.2f}, post-launch {post.card_gp_band.mean():.2f}")
print(f"NIQ Google Play charges/day post-launch: {post.niq_gp_charges.mean():.2f}")
print(f"level ratio card uplift / NIQ (post-launch, after subtracting the pre-launch card baseline): "
      f"{(post.card_gp_band.mean() - pre) / post.niq_gp_charges.mean():.2f}")
print("correlation, daily post-launch:", round(post.card_gp_band.corr(post.niq_gp_charges), 2))
print("correlation, weekly post-launch:", round(w_post.card_gp_band.corr(w_post.niq_gp_charges), 2))

card in-band/day: pre-launch 5.16, post-launch 7.58
NIQ Google Play charges/day post-launch: 2.46
level ratio card uplift / NIQ (post-launch, after subtracting the pre-launch card baseline): 0.98
correlation, daily post-launch: -0.0
correlation, weekly post-launch: 0.78


## 2. Receipt behaviour by store

In [6]:
# Muse: receipts per mailbox and the gap between a mailbox's first and second charge receipt.
m = niq[(niq.store != "Other") & (niq.kind == "charge")].drop_duplicates(["mailbox_id", "store", "order_date", "description"])
per = m.groupby(["store", "mailbox_id"]).agg(receipts=("order_date", "size"), days=("order_date", "nunique"),
                                             first=("order_date", "min"))
second = (m.sort_values("order_date").groupby(["store", "mailbox_id"]).order_date.apply(
    lambda s: (s.drop_duplicates().iloc[1] - s.iloc[0]).days if s.nunique() > 1 else None))
per["gap_days"] = second
print(per.groupby("store").agg(mailboxes=("receipts", "size"), receipts=("receipts", "sum"),
                               multi_day=("days", lambda s: (s > 1).sum()),
                               median_gap=("gap_days", "median")).to_string())
# Same-day pairs: Apple sends a $0 'subscription confirmation' beside the paid receipt for new subs.
m.assign(priced=m.item_price.notna() & (m.item_price > 0)).groupby(["store", "priced"]).size().unstack(fill_value=0)

             mailboxes  receipts  multi_day  median_gap
store                                                  
Apple              189       189          0         NaN
Google Play         75        76          1         3.0


priced,False,True
store,,
Apple,189,0
Google Play,0,76


In [7]:
# Mature benchmark: ChatGPT Plus monthly, Jul-Sep 2026. If a store sends a receipt for every renewal,
# a subscriber should show about one receipt a month, roughly 30 days apart.
cg = q("""
    SELECT i.order_date::DATE AS order_date, i.mailbox_id,
           CASE WHEN i.merchant_name ILIKE 'itunes%' THEN 'Apple' ELSE 'Google Play' END AS store,
           i.description, i.item_price
    FROM SHARE_NIQ.SRS.ITEMS_EXTENDED i
    JOIN SHARE_NIQ.SRS.MAILBOXES_ENHANCED mb ON mb.mailbox_id = i.mailbox_id AND mb.status ILIKE 'active'
    WHERE i.order_date BETWEEN '2026-07-01' AND '2026-09-30'
      AND (i.merchant_name ILIKE 'itunes' OR i.merchant_name ILIKE 'google play')
      AND i.description ILIKE '%chatgpt plus%' AND i.description NOT ILIKE '%annual%'
      AND i.item_price > 0
""")
cg["order_date"] = pd.to_datetime(cg.order_date)
cg = cg.drop_duplicates(["mailbox_id", "store", "order_date"])
cg["month"] = cg.order_date.dt.to_period("M")
months = cg.groupby(["store", "mailbox_id"]).month.nunique()
gaps = cg.sort_values("order_date").groupby(["store", "mailbox_id"]).order_date.diff().dt.days.dropna()
july = cg[cg.month == "2026-07"].groupby("store").mailbox_id.unique()
retained = {s: cg[(cg.store == s) & (cg.month == "2026-08") & cg.mailbox_id.isin(ids)].mailbox_id.nunique() / len(ids)
            for s, ids in july.items()}
pd.DataFrame({
    "receipts": cg.groupby("store").size(),
    "mailboxes": cg.groupby("store").mailbox_id.nunique(),
    "share_with_3_months": months.eq(3).groupby("store").mean().round(3),
    "median_gap_days": gaps.groupby(cg.loc[gaps.index, "store"]).median(),
    "share_gaps_25_35d": gaps.between(25, 35).groupby(cg.loc[gaps.index, "store"]).mean().round(3),
    "jul_payers_seen_in_aug": pd.Series(retained).round(3),
})

,receipts,mailboxes,share_with_3_months,median_gap_days,share_gaps_25_35d,jul_payers_seen_in_aug
Apple,124428,49811,0.601,31.0,0.826,0.840
Google Play,32083,13573,0.582,31.0,0.959,0.887


## Conclusions (run 2026-10-08)

**1. Cards vs e-receipts (Google Play).** The two are directionally consistent, but cards cannot isolate Muse.
- Google Play bills every Meta app as `GOOGLE *FACEBOOK` (several variants). There is no Muse descriptor.
- In the Muse in-app price bands ($19.99-22, $99.99-110), card transactions rose from 5.2 a day before launch to 7.6 a day after, an uplift of about 2.4 a day (+47%). NIQ Google Play Muse charges average 2.5 a day after launch.
- Weekly, the two move together: card 42 / 45 / 50 / 62 against NIQ 1 / 7 / 27 / 29 for the weeks from 09-07 (correlation 0.78 over 4 weeks). Daily there is no correlation (counts of 0-5 a day are noise).
- The near-1.0 level ratio is a coincidence of panel sizes, not a calibration.
- The uplift sits on a large pre-launch baseline of other Meta subscriptions in the same bands, so cards cannot isolate Muse. E-receipts remain the in-app source. This corrects the earlier note that Google Play showed "no launch uplift": there is a small one.

**2. Store receipt behaviour.**
- **Both stores send a priced receipt for every renewal.** For ChatGPT Plus monthly (Jul-Sep 2026, active mailboxes), the median gap between receipts is 31 days for both stores, and 96% (Google Play) and 83% (Apple) of gaps fall within 25-35 days. Of July payers, 89% (Google Play) and 84% (Apple) have an August receipt, consistent with normal churn.
- So Muse e-receipt charges after month 1 will include renewals from both stores. Muse's first renewals fall due from about 2026-10-08. Before then, every Muse charge is a first payment, which is why gross adds equal charges today.
- **Apple sends two emails at sign-up, and the dashboard currently counts the wrong one.**
  - Apple sends an unpriced subscription confirmation (`Muse from Meta Power Plan $20.00 /month`). All 189 Apple Muse "charges" in the app are these confirmations.
  - Apple's priced receipt does not name the app: the description is just `Power Plan` / `Maximum Plan` (merchant iTunes, $20 / $100).
  - Those receipts start in launch week and are absent before it: 5 / 15 / 36 / 73 a week from 09-07 for Power, 8 in total for Max.
  - The app's `'%muse from meta%'` filter misses them. Until it is fixed, the Apple series measures new subscriptions (confirmations), not payments, and will not show Apple renewals.
  - Google Play receipts name the app and carry the price, so they are unaffected.
- Recommended fix: add iTunes `Power Plan` / `Maximum Plan` as Apple `charge` rows, and class the unpriced confirmations as a new `signup` kind. Apple gross adds then come from confirmations, and Apple charges from the priced receipts.